In [ ]:
pip install ollama

In [ ]:
pip install langchain langchain-openai pydantic python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 16.3 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28


In [ ]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (583 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently i

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
%%bash --bg
ollama serve

In [ ]:
!mkdir -p /content/drive/MyDrive/ollama_models
!rm -rf ~/.ollama/models
!mkdir -p ~/.ollama
!ln -s /content/drive/MyDrive/ollama_models ~/.ollama/models

In [ ]:
!rm -rf ~/.ollama/models
!mkdir -p /content/ollama_models
!ln -s /content/ollama_models ~/.ollama/models


In [ ]:
import subprocess
import time

In [ ]:
process = subprocess.Popen(["ollama", "serve"])

In [ ]:
!ollama pull mistral-nemo:12b
!ollama pull qwen2:7b
!ollama pull llama3.1:8b
!ollama pull gemma2:9b
!ollama pull deepseek-r1:8b
!ollama pull phi3:3.8b


In [ ]:
!ollama list

NAME                ID              SIZE      MODIFIED               
phi3:3.8b           4f2222927938    2.2 GB    Less than a second ago    
deepseek-r1:8b      6995872bfe4c    5.2 GB    18 seconds ago            
gemma2:9b           ff02c3702f32    5.4 GB    55 seconds ago            
llama3.1:8b         46e0c10c039e    4.9 GB    About a minute ago        
qwen2:7b            dd314f039b9d    4.4 GB    2 minutes ago             
mistral-nemo:12b    e7e06d107c6c    7.1 GB    3 minutes ago             


In [ ]:
TEAM = [
    "qwen2:7b",
    "mistral-nemo:12b",
    "gemma2:9b",
    "llama3.1:8b",
    "deepseek-r1:8b",
    "phi3:3.8b"
]

In [ ]:
!ollama list

NAME                ID              SIZE      MODIFIED               
phi3:3.8b           4f2222927938    2.2 GB    Less than a second ago    
deepseek-r1:8b      6995872bfe4c    5.2 GB    18 seconds ago            
gemma2:9b           ff02c3702f32    5.4 GB    55 seconds ago            
llama3.1:8b         46e0c10c039e    4.9 GB    About a minute ago        
qwen2:7b            dd314f039b9d    4.4 GB    2 minutes ago             
mistral-nemo:12b    e7e06d107c6c    7.1 GB    3 minutes ago             


In [ ]:
!ollama rm llama3:8b

Error: model 'llama3:8b' not found


In [ ]:
import ollama
import json
import time
import math
import csv
import os
from collections import Counter
from datetime import datetime

# =============================
# CONFIG
# =============================

TEAM = [
    "qwen2:7b",
    "mistral-nemo:12b",
    "gemma2:9b",
    "llama3.1:8b",
    "deepseek-r1:8b",
    "phi3:3.8b"
]

SYSTEM_PROMPT = "Answer concisely. Only give the answer."
LOG_FILE = "experiment_log.jsonl"

# =============================
# UTIL
# =============================

def normalize(text):
    return text.lower().strip()

def load_existing():
    results = {}
    if not os.path.exists(LOG_FILE):
        return results

    with open(LOG_FILE, "r") as f:
        for line in f:
            entry = json.loads(line)
            results[entry["question_id"]] = entry
    return results

def save_entry(entry):
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(entry) + "\n")

def get_last_completed_index():
    existing = load_existing()
    if not existing:
        return 0
    return max(existing.keys()) + 1

# =============================
# MODEL CALL
# =============================

def ask_model(model, question, temperature=0.2):
    start = time.time()

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ],
        options={"temperature": temperature}
    )

    latency = time.time() - start

    answer = response["message"]["content"].strip()

    return {
        "model": model,
        "answer_raw": answer,
        "answer_norm": normalize(answer),
        "latency": latency
    }

# =============================
# RUN ONE QUESTION
# =============================

def run_question(question, true_answer=None):

    answers = []

    for model in TEAM:
        print(f"  → {model}")
        result = ask_model(model, question)
        answers.append(result)

    votes = Counter(a["answer_norm"] for a in answers)
    final_answer = votes.most_common(1)[0][0]

    entry = {
        "question_id": None,  # filled later
        "timestamp": datetime.utcnow().isoformat(),
        "question": question,
        "true_answer": true_answer,
        "individual_answers": answers,
        "vote_distribution": dict(votes),
        "final_answer": final_answer,
        "agreement_rate": max(votes.values()) / len(TEAM),
        "entropy": -sum(
            (c/len(TEAM)) * math.log2(c/len(TEAM))
            for c in votes.values()
        )
    }

    if true_answer:
        entry["correct"] = normalize(true_answer) == final_answer

    return entry

# =============================
# RUN FROM CSV (FLEXIBLE)
# =============================

def run_experiment(
    csv_path,
    start_index=None,
    limit=None,
    question_column="question",
    answer_column=None
):

    existing = load_existing()

    if start_index is None:
        start_index = get_last_completed_index()

    with open(csv_path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    total = len(rows)
    end_index = total if limit is None else min(start_index + limit, total)

    for i in range(start_index, end_index):

        if i in existing:
            print(f"Skipping Q{i} (already done)")
            continue

        row = rows[i]
        question = row[question_column]
        true_answer = row[answer_column] if answer_column else None

        print(f"\nRunning Q{i}: {question[:60]}")

        entry = run_question(question, true_answer)
        entry["question_id"] = i

        save_entry(entry)

    print("\nDone.")

In [ ]:
process = subprocess.Popen(["ollama", "serve"])

In [ ]:
run_experiment(
    csv_path="replacement_trivia_questions.csv",
    start_index=0,
    limit=1000,
    answer_column="answer"
)


Running Q0: The body's 3 major types of blood vessels are the arteries, 
  → qwen2:7b
  → mistral-nemo:12b
  → gemma2:9b
  → llama3.1:8b
  → deepseek-r1:8b
  → phi3:3.8b


/tmp/ipykernel_13688/56316783.py:99: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),



Running Q1: Pashtun leader & nonviolence advocated Abdul Ghaffar Khan wa
  → qwen2:7b
  → mistral-nemo:12b
  → gemma2:9b
  → llama3.1:8b
  → deepseek-r1:8b
  → phi3:3.8b

Running Q2: (I'm Louis C.K.) In a 2015 film I played Arlen Hird, a compo
  → qwen2:7b
  → mistral-nemo:12b
  → gemma2:9b
  → llama3.1:8b
  → deepseek-r1:8b
  → phi3:3.8b

Running Q3: Legend says this capital is named for Kii, one of the 3 brot
  → qwen2:7b
  → mistral-nemo:12b
  → gemma2:9b
  → llama3.1:8b
  → deepseek-r1:8b
  → phi3:3.8b

Running Q4: A broken-down old nag is called a Rosinante in honor of this
  → qwen2:7b
  → mistral-nemo:12b
  → gemma2:9b
  → llama3.1:8b
  → deepseek-r1:8b
  → phi3:3.8b

Done.
